In [44]:
# 뉴스 raw 데이터를 DB에 넣을려고하는데 
# 이미 수집된 기사들(csv)에 신문사 이름 넣는 노트북임

import pandas as pd

chosun_ilbo = pd.read_csv("../data/raw/news/chosun_ilbo.csv")
dong_a_ilbo = pd.read_csv("../data/raw/news/dong_a_ilbo.csv")
joongang_ilbo = pd.read_csv("../data/raw/news/joongang_ilbo.csv")
korea_economy = pd.read_csv("../data/raw/news/korea_economy.csv")


In [47]:
import pymysql

In [45]:

chosun_ilbo['publisher'] = '조선일보'

dong_a_ilbo['publisher'] = '동아일보'
joongang_ilbo['publisher'] = '중앙일보'
korea_economy['publisher'] = '한국경제'

chosun_ilbo.head()

,URL,content,publisher
0,https://www.chosun.com/economy/real_estate/202...,불황기 재테크의 최고 수단을 떠올리면 부동산 경매가 단연 첫손가락에 꼽힙니다. 그러...,조선일보
1,https://www.chosun.com/economy/real_estate/202...,"최근 전·월세 금액이 급등하며 세입자들의 주거비 부담이 늘어나는 가운데, 집주인이 ...",조선일보
2,https://www.chosun.com/economy/real_estate/202...,현대건설이 올 들어 정비사업 누적 수주액 3조원을 돌파했다. 현대건설은 지난 30일...,조선일보
3,https://www.chosun.com/economy/real_estate/202...,주택담보대출 금리가 7년 7개월 만에 최고치로 치솟았다. 금리 인상 여파로 집을 사...,조선일보
4,https://www.chosun.com/economy/real_estate/202...,공인중개사 전용 프로그램 ‘땅집고리얼터’는 다음 달 8일 재건축·재개발 주택을 매물...,조선일보


In [73]:
chosun_ilbo.to_csv('../data/raw/news/chosun_ilbo.csv', index=False)
dong_a_ilbo.to_csv('../data/raw/news/dong_a_ilbo.csv', index=False)
joongang_ilbo.to_csv('../data/raw/news/joongang_ilbo.csv', index=False)
korea_economy.to_csv('../data/raw/news/korea_economy.csv', index=False)

In [74]:
chosun_ilbo.isna().sum()

URL          0
content      0
publisher    0
dtype: int64

In [70]:
chosun_ilbo.dropna(inplace=True)

In [78]:
news_list = ['chosun_ilbo', 'dong_a_ilbo', 'joongang_ilbo', 'korea_economy']

try:
    # 1) MySQL 연결
    conn = pymysql.connect(
        host='localhost',
        user='root',
        password='As589788@@',
        db='apt_price',
        charset='utf8mb4'
    )

    with conn.cursor() as cursor:
        for news in news_list:
            # 2) 신문사별 CSV 읽기
            csv_file = f"../data/raw/news/{news}.csv"
            df = pd.read_csv(csv_file)

            # 3) INSERT 쿼리
            sql = """
                INSERT INTO news (url, content, publisher)
                VALUES (%s, %s, %s)
            """

            # 4) 데이터프레임 행 반복하며 INSERT
            for idx, row in df.iterrows():
                cursor.execute(sql, (row['URL'], row['content'], row['publisher']))

        # 5) 모든 INSERT 후 커밋
        conn.commit()

except Exception as e:
    print("DB 연결 실패 혹은 처리 오류:", e)

finally:
    conn.close()